# CLASS PopularityRecommender

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
# Mount Drive
drive.mount('/content/drive')

# Load dữ liệu
file_path = '/content/drive/MyDrive/datasets/data_train.csv'
df = pd.read_csv(file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

class PopularityRecommender:
    """
    Baseline recommender chỉ dựa trên popularity của bài hát.
    Gợi ý Top-N bài hát phổ biến nhất cho mọi user (non-personalized).
    """

    def __init__(self):
        self.song_popularity = None      # Series: song_id -> mean popularity
        self.top_songs = None            # List song_id được sắp xếp theo popularity giảm dần
        self.item_to_title = None        # Dict song_id -> title
        self.item_to_artist = None       # Dict song_id -> artist_name

    def fit(self, df: pd.DataFrame, item_col='track_id', popularity_col='popularity',
            title_col='title', artist_col='artist_name'):
        """
        Huấn luyện model từ dữ liệu.

        Parameters:
        - df: DataFrame chứa ít nhất các cột item_col và popularity_col
        - item_col: tên cột chứa ID bài hát (mặc định 'track_id')
        - popularity_col: tên cột popularity (mặc định 'popularity')
        - title_col, artist_col: dùng để hiển thị thông tin bài hát khi recommend
        """
        print("Đang tính popularity trung bình cho từng bài hát...")

        # Tính mean popularity theo item (vì một bài hát có thể xuất hiện nhiều lần)
        self.song_popularity = df.groupby(item_col)[popularity_col].mean()

        # Sort giảm dần để có top songs
        self.top_songs = self.song_popularity.sort_values(ascending=False).index.tolist()

        # Lưu mapping để hiển thị tên bài hát và nghệ sĩ (lấy giá trị đầu tiên là đủ)
        self.item_to_title = df.drop_duplicates(item_col).set_index(item_col)[title_col].to_dict()
        self.item_to_artist = df.drop_duplicates(item_col).set_index(item_col)[artist_col].to_dict()

        print(f"Đã huấn luyện xong! Có {len(self.top_songs)} bài hát duy nhất.")
        print(f"Top 5 bài hát phổ biến nhất:")
        self.print_recommendations(self.top_songs[:5])

    def recommend(self, n: int = 10):
        """
        Gợi ý Top-N bài hát phổ biến nhất.

        Returns:
        - List track_id
        """
        if self.top_songs is None:
            raise ValueError("Model chưa được fit. Gọi fit() trước khi recommend.")

        return self.top_songs[:n]

    def recommend_with_details(self, n: int = 10):
        """
        Gợi ý Top-N kèm thông tin chi tiết (title, artist, popularity).

        Returns:
        - DataFrame với các cột: rank, track_id, title, artist_name, popularity
        """
        if self.top_songs is None:
            raise ValueError("Model chưa được fit. Gọi fit() trước khi recommend.")

        top_n_ids = self.top_songs[:n]

        details = pd.DataFrame({
            'track_id': top_n_ids,
            'title': [self.item_to_title.get(sid, 'Unknown') for sid in top_n_ids],
            'artist_name': [self.item_to_artist.get(sid, 'Unknown') for sid in top_n_ids],
            'popularity': [self.song_popularity.get(sid, np.nan) for sid in top_n_ids]
        })
        details.insert(0, 'rank', range(1, len(details) + 1))

        return details

    def print_recommendations(self, track_ids):
        """In ra thông tin các bài hát một cách đẹp."""
        for i, sid in enumerate(track_ids, 1):
            title = self.item_to_title.get(sid, 'Unknown')
            artist = self.item_to_artist.get(sid, 'Unknown')
            pop = self.song_popularity.get(sid, np.nan)
            print(f"{i:2d}. {title} - {artist} (popularity: {pop:.1f})")

In [ ]:
# Khởi tạo và huấn luyện model
pop_recommender = PopularityRecommender()

pop_recommender.fit(
    df=df,
    item_col='track_id',
    popularity_col='popularity',
    title_col='title',
    artist_col='artist_name'
)

# Gợi ý Top-10 (chỉ list track_id)
top_10_ids = pop_recommender.recommend(n=10)
print("Top 10 track_id:", top_10_ids)

# Gợi ý Top-10 kèm thông tin chi tiết
recommendations = pop_recommender.recommend_with_details(n=10)
print("\nTop 10 bài hát phổ biến nhất:")
print(recommendations)

# In đẹp ra console
print("\nGợi ý cho bất kỳ user nào:")
pop_recommender.print_recommendations(top_10_ids)

Đang tính popularity trung bình cho từng bài hát...
Đã huấn luyện xong! Có 11256 bài hát duy nhất.
Top 5 bài hát phổ biến nhất:
 1. Rockin' Around The Christmas Tree - Brenda Lee (popularity: 85.0)
 2. In The End (Album Version) - Linkin Park (popularity: 84.0)
 3. Yellow - Coldplay (popularity: 84.0)
 4. The Scientist - Coldplay (popularity: 84.0)
 5. Yellow (Live In Sydney) - Coldplay (popularity: 84.0)
Top 10 track_id: ['TRVQLGY12903CB348F', 'TRJCWTZ128E0797FCA', 'TRIKGRK128E0780DB0', 'TRQFXKD128E0780CAE', 'TRLNWKC128E0780CD8', 'TRUOCPQ128F1480E90', 'TRLQTMK128E07810A3', 'TRBZGSM128E078EDB4', 'TRVRDXR128F423A018', 'TRVCUSW128F92F20C6']

Top 10 bài hát phổ biến nhất:
   rank            track_id                              title  artist_name  \
0     1  TRVQLGY12903CB348F  Rockin' Around The Christmas Tree   Brenda Lee   
1     2  TRJCWTZ128E0797FCA         In The End (Album Version)  Linkin Park   
2     3  TRIKGRK128E0780DB0                             Yellow     Coldplay   
3     

# Class Evaluate

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from typing import Dict, List

class RecommendationEvaluator:
    """
    Đánh giá Top-N Recommendation: Precision@K, Recall@K, F1@K, HitRate@K, NDCG@K
    """

    def __init__(self):
        self.metrics = None  # Sẽ lưu DataFrame summary

    def _precision_at_k(self, rec_list: List, ground_truth: List, k: int) -> float:
        rec_set = set(rec_list[:k])
        gt_set = set(ground_truth)
        correct = len(rec_set & gt_set)
        return correct / k

    def _recall_at_k(self, rec_list: List, ground_truth: List, k: int) -> float:
        gt_set = set(ground_truth)
        if len(gt_set) == 0:
            return 0.0
        rec_set = set(rec_list[:k])
        correct = len(rec_set & gt_set)
        return correct / len(gt_set)

    def _f1_at_k(self, precision: float, recall: float) -> float:
        if precision + recall == 0:
            return 0.0
        return 2 * (precision * recall) / (precision + recall)

    def _hit_rate_at_k(self, rec_list: List, ground_truth: List, k: int) -> float:
        rec_set = set(rec_list[:k])
        gt_set = set(ground_truth)
        return 1.0 if rec_set & gt_set else 0.0

    def _ndcg_at_k(self, rec_list: List, ground_truth: List, k: int) -> float:
        gt_set = set(ground_truth)
        relevance = [1.0 if item in gt_set else 0.0 for item in rec_list[:k]]

        if sum(relevance) == 0:
            return 0.0

        dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
        idcg = sum(1 / np.log2(idx + 2) for idx in range(min(len(gt_set), k)))
        return dcg / idcg if idcg > 0 else 0.0

    def evaluate(self, test_data: Dict[str, List], recommendations: Dict[str, List],
                 k_values: List[int] = [5, 10, 20]) -> pd.DataFrame:
        """
        Đánh giá toàn bộ hệ thống và trả về summary table (mean ± std)
        """
        results = []

        for user_id in test_data:
            if user_id not in recommendations:
                continue
            rec_list = recommendations[user_id]
            gt_list = test_data[user_id]

            user_row = {'user_id': user_id}
            for k in k_values:
                p = self._precision_at_k(rec_list, gt_list, k)
                r = self._recall_at_k(rec_list, gt_list, k)
                f1 = self._f1_at_k(p, r)
                hr = self._hit_rate_at_k(rec_list, gt_list, k)
                ndcg = self._ndcg_at_k(rec_list, gt_list, k)

                user_row[f'Precision@{k}'] = p
                user_row[f'Recall@{k}'] = r
                user_row[f'F1@{k}'] = f1
                user_row[f'HitRate@{k}'] = hr
                user_row[f'NDCG@{k}'] = ndcg

            results.append(user_row)

        df_results = pd.DataFrame(results)

        # Tính mean ± std và lưu dưới dạng số (không phải string)
        summary_rows = []
        for k in k_values:
            row = {}
            for metric in ['Precision', 'Recall', 'F1', 'HitRate', 'NDCG']:
                col = f'{metric}@{k}'
                if col in df_results.columns:
                    mean_val = df_results[col].mean()
                    std_val = df_results[col].std()
                    row[col + '_mean'] = mean_val
                    row[col + '_std'] = std_val
                    row[col] = f"{mean_val:.4f} ± {std_val:.4f}"
            summary_rows.append(row)

        summary_df = pd.DataFrame(summary_rows, index=k_values)
        self.metrics = summary_df
        self.detailed_results = df_results  # Lưu lại để debug nếu cần

        return summary_df

    def print_summary(self, k_values: List[int] = None):
        """In kết quả đánh giá đẹp mắt"""
        if self.metrics is None:
            print("Chưa chạy evaluate()")
            return

        if k_values is None:
            k_values = self.metrics.index.tolist()

        print("EVALUATION SUMMARY (Popularity Baseline)")
        print("=" * 70)

        # Chỉ hiển thị các cột formatted (có ±)
        display_cols = [col for col in self.metrics.columns if '±' in str(self.metrics[col].iloc[0])]
        print(self.metrics[display_cols].loc[k_values])

        # Tìm best K theo F1 trung bình
        f1_means = []
        for k in k_values:
            col = f'F1@{k}'
            if col in self.metrics.columns:
                mean_str = self.metrics.loc[k, col].split(' ± ')[0]
                f1_means.append(float(mean_str))
            else:
                f1_means.append(0.0)

        if f1_means:
            best_k = k_values[np.argmax(f1_means)]
            best_f1 = self.metrics.loc[best_k, f'F1@{best_k}']
            print(f"\nBest performance theo F1: F1@{best_k} = {best_f1}")
        print("=" * 70)

# ĐÁNH GIÁ

In [ ]:
# 1. Tạo recommendations từ model
def get_recommendations_for_users_fixed(recommender, user_ids, n=50):
    """Recommend 50 songs để cover K=20"""
    top_songs = recommender.recommend(n=50)  # Lấy 50 thay vì 20
    return {user_id: top_songs for user_id in user_ids}


# 2. Chạy evaluation
test_path = '/content/drive/MyDrive/datasets/data_test.csv'
test_data = pd.read_csv(test_path)

pop_recommender.fit(df)

# Tạo recommendations
user_recommendations_fixed = get_recommendations_for_users_fixed(pop_recommender, test_data.keys(), n=50)

# Đánh giá
evaluator_fixed = RecommendationEvaluator()
summary_fixed = evaluator_fixed.evaluate(test_data, user_recommendations_fixed, k_values=[5, 10, 20])
evaluator_fixed.print_summary()

Đang tính popularity trung bình cho từng bài hát...
Đã huấn luyện xong! Có 11256 bài hát duy nhất.
Top 5 bài hát phổ biến nhất:
 1. Rockin' Around The Christmas Tree - Brenda Lee (popularity: 85.0)
 2. In The End (Album Version) - Linkin Park (popularity: 84.0)
 3. Yellow - Coldplay (popularity: 84.0)
 4. The Scientist - Coldplay (popularity: 84.0)
 5. Yellow (Live In Sydney) - Coldplay (popularity: 84.0)
EVALUATION SUMMARY (Popularity Baseline)
        Precision@5         Recall@5             F1@5        HitRate@5  \
5   0.0400 ± 0.2000  0.0000 ± 0.0001  0.0000 ± 0.0002  0.0400 ± 0.2000   
10              NaN              NaN              NaN              NaN   
20              NaN              NaN              NaN              NaN   

             NDCG@5  
5   0.0400 ± 0.2000  
10              NaN  
20              NaN  

Best performance theo F1: F1@10 = 0.0001 ± 0.0004
